In [ ]:
!pip install hf_xet

In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [ ]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"Using device: {device}")

Using device: cuda


Load a model and tokenizer for the SmolLM2 small model from HuggingFace 🤗

In [ ]:
model_name="HuggingFaceTB/SmolLM2-360M-Instruct"


In [ ]:
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/724M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

The tokenizer converts a string to a 2D tensor of tokens.

In [ ]:
model_inputs = tokenizer.encode("A list of colors: red, blue")
model_inputs


[49, 1398, 282, 4683, 42, 2382, 28, 4461]

We can decode these token IDs one by one using the tokenizer `decode` method

In [ ]:
[tokenizer.decode(token_id) for token_id in model_inputs]

['A', ' list', ' of', ' colors', ':', ' red', ',', ' blue']

To generate a continuation of a prompt, we tokenize the prompt using PyTorch tensors, and feed this into the model created above.

In [ ]:
model_inputs = tokenizer(["A sequence of numbers: 2, 4, 8, 14"], return_tensors="pt").to(device)
print(model_inputs)
generated_ids = model.generate(**model_inputs, max_new_tokens=20)
print(generated_ids)
tokenizer.decode(generated_ids[0], skip_special_tokens=True)

## Looking at the logits

Request a continuation from the model, with `output_logits=True`,and `return_dict_in_generate=True`. Note you need both options.

In [ ]:
model_inputs = tokenizer(["Continue the sequence of numbers: 2, 4, 8"], return_tensors="pt").to(device)
model_output = model.generate(**model_inputs, max_new_tokens=20, do_sample=False, output_logits=True, return_dict_in_generate=True)

The output token IDs are contained in `model_output.sequences`

In [ ]:
model_output.sequences[0]

tensor([26226,   260,  5972,   282,  2966,    42,   216,    34,    28,   216,
           36,    28,   216,    40,    28,   216,    33,    38,    28,   216,
           35,    34,    28,   216,    38,    36,    28,   216,    33,    34,
           40,    28,   216,    34])

The output logits for each step are contained in the tuple `model_output.logits` each entry is a tensor of $\text{bach_dim} \times \text{vocab_size}$ dimensions

In [ ]:
model_output.logits[0].shape

torch.Size([1, 49152])

The output can be reconstructed from the token IDs using tokenizer functions

In [ ]:
tokenizer.decode(model_output.sequences[0])

'Continue the sequence of numbers: 2, 4, 8, 16, 32, 64, 128, 2'

Let's pack the output logits into a numpy array

In [ ]:
output_len = len(model_output.logits)
output_logits = np.array([model_output.logits[k][0].cpu() for k in range(output_len)])

The greedy-decoding results is the maximum logit at each postion in the sequence (as we are using greedy decoding), namely the maximum index over the second (token) axis




In [ ]:
output_tokens = [tokenizer.decode(t) for t in output_logits.argmax(axis=1)]
"".join(output_tokens)

', 16, 32, 64, 128, 2'

The greedy-decoding results is the maximum logit at each postion in the sequence (as we are using greedy decoding), namely the maximum index over the second (token) axis

In [ ]:
output_token_ids = output_logits.argmax(axis=1)
np.allclose(output_token_ids, model_output.sequences[0, -output_len:].cpu())

True

In [ ]:
def show_logits_with_color_scale(model_output, logits_processor=None, num_top_tokens=15):
    def color_scale(val, min_val, max_val):
        if isinstance(val, str):
            return ''

        # Normalize the value to the range [0, 1]
        normalized_val = (val - min_val) / (max_val - min_val) if (max_val - min_val) != 0 else 0

        # Calculate the RGB values based on the normalized value
        # Transition from blue to red
        red = int(255 * normalized_val)
        blue = int(255 * (1 - normalized_val))
        green = int(150 * normalized_val)

        # Highlight if max value
        return f'background-color: rgb({red}, {green}, {blue});'

    def maximum_value_in_column(column):
        highlight = 'color: white; font-weight: bold'
        default = ''
        maximum_in_column = column.max()

        # must return one string per cell in this column
        return [highlight if v == maximum_in_column else default for v in column]

    output_len = len(model_output.logits)
    input_len = len(model_output.sequences[0]) - output_len
    if logits_processor:
        output_logits = np.array([logits_processor(model_output.sequences[:,:input_len+k], model_output.logits[k])[0].cpu() for k in range(output_len)])
    else:
        output_logits = np.array([model_output.logits[k][0].cpu() for k in range(output_len)])

    output_idx = output_logits.argmax(axis=1)
    output_tokens = [tokenizer.decode(t) for t in output_idx]

    logits_idx_top = output_logits.max(axis=0).argsort()[-num_top_tokens:]
    logits_idx_top = sorted([int(idx) for idx in set(logits_idx_top).union(output_idx)])

    token_str_top = [tokenizer.decode(t) for t in logits_idx_top]
    column_index = pd.MultiIndex.from_tuples([(i, token) for i, token in enumerate(output_tokens)], names=['index', 'token'])

    df = pd.DataFrame(output_logits[:, logits_idx_top].T, index=[repr(s.encode('utf-8'))[1:] for s in token_str_top], columns=column_index)

    # Apply the styling
    styled_df = df.style.map(lambda x: color_scale(x, df.min().min(), df.max().max()), subset=df.columns)
    styled_df.apply(maximum_value_in_column, axis=0)

    # Format values to two decimal places
    styled_df = styled_df.format("{:.2f}")

    return styled_df

It's interesting to compare the logits for a sequence that it gets right, and notice that the difference between highest and next highest answer are considerably higher for the case where it has the right answer (the more known sequence), especially for the second digit

In [ ]:
model_inputs = tokenizer(["Continue the sequence of numbers: 2, 4, 8"], return_tensors="pt").to(device)
model_output = model.generate(**model_inputs, max_new_tokens=20, do_sample=False, output_logits=True, return_dict_in_generate=True)

In [ ]:
show_logits_with_color_scale(model_output)

index,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
token,",",,1,6,",",,3,2,",",,6,4,",",,1,2,8,",",,2
"','",19.67,5.11,3.35,16.16,19.04,3.05,4.96,15.65,18.22,3.53,6.14,14.22,18.68,3.62,6.27,15.57,14.46,17.90,4.02,3.99
'0',9.31,5.33,11.49,15.32,7.27,6.05,10.68,15.21,7.80,4.95,10.18,15.88,6.25,5.06,10.33,19.07,15.83,6.70,5.09,11.32
'1',10.45,11.12,20.98,14.22,7.67,9.40,14.90,16.75,7.55,7.49,15.04,15.40,6.28,7.55,22.78,13.36,14.31,6.31,7.93,15.61
'2',8.69,6.84,14.89,16.27,6.81,8.11,15.83,23.21,6.05,5.79,12.29,18.58,5.45,5.63,11.93,24.29,13.65,5.36,6.20,20.62
'3',8.38,7.27,16.63,14.27,6.07,8.00,21.49,14.41,5.70,6.42,11.70,18.94,5.17,6.07,10.13,11.91,12.59,4.78,6.54,14.34
'4',9.87,5.67,13.39,17.31,7.15,6.86,14.81,14.36,6.75,5.88,13.47,27.06,6.31,5.73,10.78,11.34,14.79,6.07,5.96,13.96
'5',8.52,5.87,13.02,15.86,5.71,7.13,12.78,12.81,5.53,6.05,14.67,17.91,5.86,6.32,10.27,11.43,16.62,4.71,5.80,14.30
'6',9.12,6.07,14.33,22.83,6.14,6.65,16.26,14.77,6.02,5.68,21.26,12.55,6.49,6.05,9.25,11.61,16.30,5.20,5.78,10.56
'7',9.19,6.64,14.39,15.48,6.91,7.92,11.19,12.60,5.86,6.57,12.46,13.26,5.09,6.87,10.21,6.87,19.87,4.97,6.28,9.72


## Logit Processors

Let's try some logit processing. Tis code is modified from (ReLLM)[https://github.com/r2d4/rellm] and supresses tokens not in `allowed_token_ids` from being generated by subtracting a number from their logits.

In [ ]:
from transformers import LogitsProcessor

class LogitsMask(LogitsProcessor):
    def __init__(self, allowed_token_ids):
        self.allowed_token_ids = set(allowed_token_ids)

    def __call__(self, input_ids, scores):
        mask = torch.ones_like(scores) * -10
        for token_id in self.allowed_token_ids:
            mask[:, token_id] = 0
        scores = scores + mask
        return scores

Let's define numeric tokens that we want the model to preference. Note that the SmolLM2 model tokenizer converts individual numbers to tokens and there are no multi-character tokens containing numbers. We can confirm this by searching the entire vocab for tokens with numbers in them:

In [ ]:
import regex
re_num = regex.compile(r".*(\d+).*")
[v for v in tokenizer.get_vocab().keys() if re_num.match(v)]


['9', '6', '3', '0', '1', '4', '7', '8', '5', '2']

Let's allow only even numbers and see how the output changes

In [ ]:
allowed_tokens = list("0123456789 ")
allowed_token_ids = [tokenizer.encode(t)[0] for t in allowed_tokens] + [tokenizer.eos_token_id]
lm_test = LogitsMask(allowed_token_ids)

In [ ]:
allowed_token_ids

[32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 216, 2]

In [ ]:
model_inputs = tokenizer(["Continue the sequence of numbers: 2, 4, 8"], return_tensors="pt").to(device)
model_output = model.generate(**model_inputs, max_new_tokens=20, logits_processor=[lm_test], do_sample=False, output_logits=True, return_dict_in_generate=True)

In [ ]:
show_logits_with_color_scale(model_output, lm_test)

index,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
token,,1,6,,3,2,,6,4,,1,2,8,,2,5,6,,5,1
'<|im_end|>',5.93,9.98,1.71,7.84,0.17,-2.24,10.71,3.17,-3.07,10.46,1.23,-0.79,-4.37,11.99,5.77,-2.10,-2.04,12.51,-0.34,-3.09
"','",9.67,-5.81,5.83,8.25,-1.80,3.21,5.50,-2.35,3.27,4.60,-2.93,3.65,2.72,3.92,-5.75,2.34,2.18,4.24,-7.27,-0.87
'.',4.36,-9.65,4.83,4.47,-4.02,1.10,5.48,-1.31,1.40,5.43,-1.09,1.68,0.97,5.16,-4.54,2.23,1.24,5.70,-3.33,0.86
'0',9.31,12.35,16.52,7.72,11.51,14.82,8.23,9.72,14.82,6.78,8.52,16.83,13.77,7.18,9.88,14.10,13.87,7.43,9.69,18.52
'1',10.45,14.59,13.98,8.72,15.64,16.45,8.97,16.45,13.89,7.11,18.68,11.58,12.29,7.31,13.31,13.96,12.59,7.28,12.31,25.96
'2',8.69,13.71,16.42,7.83,17.55,21.26,7.11,12.66,16.73,6.26,10.18,21.51,11.95,6.26,18.21,13.11,13.90,6.84,11.07,14.86
'3',8.38,13.99,14.02,8.00,20.35,14.01,7.70,11.47,17.29,6.28,8.54,11.10,11.10,5.73,13.11,11.62,14.02,5.71,12.06,12.50
'4',9.87,13.89,14.41,8.00,14.39,13.39,7.84,12.40,24.50,6.72,8.87,10.45,12.71,6.43,12.40,13.57,13.52,6.17,14.53,12.49
'5',8.52,12.94,14.66,6.90,12.83,12.44,6.95,12.01,16.80,6.72,8.20,10.71,13.96,5.29,11.48,24.00,17.59,5.58,19.00,11.82


## Constraining to a RegEx

We want to test at generation step if the generated string up to now appended by each token string in the vocabulary would be valid.

Tchis requires a partial match, where the regex processor returns valid if the supplied string is consistent with the RegEx pattern so far.  This is provided in the python regex library (but not the inbuilt re library).

RegEx is based on [this thread](https://stackoverflow.com/questions/3809401/what-is-a-good-regular-expression-to-match-a-url#3809435)

In [ ]:
import regex
regex_http = regex.compile(" ?https?:\/\/(www\.)?\w{1,256}\.\w{1,6}(?:(/|\\?|#)[^\\s]*)?$")

print(regex_http.match("http://test.com/sdas"))
print(regex_http.match("http://test."))
print(regex_http.match("http://test.", partial=True))
print(regex_http.match("http://test.$$$", partial=True))

<regex.Match object; span=(0, 20), match='http://test.com/sdas'>
None
<regex.Match object; span=(0, 12), match='http://test.', partial=True>
None


In [ ]:
print(regex_http.match("http://test.com/sdas\ntest"))


None


Let's try a parser that's given the string up to this point, and multiple tokens, and then uses regex to check the tokens that would complete the string

In [ ]:
from transformers import LogitsProcessor

class RegexMask(LogitsProcessor):
    def __init__(self, regex, tokenizer, stop_on_match=False, subtract_on_mask=10):
        self.regex = regex
        self.tokenizer = tokenizer
        self.vocab_map = {token_id: tokenizer.decode(token_id) for _, token_id in tokenizer.get_vocab().items()}
        self.first_in_sequence = None
        self.stop_on_match = stop_on_match
        self.subtract_on_mask = subtract_on_mask
        self.debug = False

    def reset(self):
        self.first_in_sequence = None

    def __call__(self, input_ids, scores):
        if self.first_in_sequence is None:
            self.first_in_sequence = input_ids.shape[1]
            partial_output_str = ''
        else:
            partial_output_str = self.tokenizer.decode(input_ids[0, self.first_in_sequence:])

        # Always allow the model to finish (even if not valid)
        allowed_token_ids = []

        if self.debug:
            print(f"current: {partial_output_str}")

        # Search through all tokens to see which could give a match to the regex
        for token_id, token_str in self.vocab_map.items():
            if self.regex.match(partial_output_str + token_str, partial=True):
                allowed_token_ids.append(token_id)

        # Force the model to finish when valid
        if self.stop_on_match and self.regex.match(partial_output_str):
            allowed_token_ids = [self.tokenizer.eos_token_id]
        else:
            allowed_token_ids += [self.tokenizer.eos_token_id]

        if self.debug:
            print(f"num allowed tokens: {len(allowed_token_ids)}")

        # Mask scores for tokens that don't match regex
        mask = -self.subtract_on_mask * torch.ones_like(scores)
        for token_id in allowed_token_ids:
            mask[:, token_id] = 0
        scores = scores + mask
        return scores

Example

In [ ]:
prompt = ["One of the most common web addresses is:"]

In [ ]:
logits_http_mask = RegexMask(regex_http, tokenizer, stop_on_match=True)
model_inputs = tokenizer(prompt, return_tensors="pt").to(device)
model_output = model.generate(**model_inputs, max_new_tokens=15, do_sample=False, output_logits=True, return_dict_in_generate=True)
decoded_output = tokenizer.batch_decode(model_output.sequences, skip_special_tokens=True)[0]
print(f"--------\n{decoded_output}\n--------")

--------
One of the most common web addresses is:

www.google.com

This is a well-known
--------


In [ ]:
logits_http_mask = RegexMask(regex_http, tokenizer, stop_on_match=True)
model_inputs = tokenizer(prompt, return_tensors="pt").to(device)
model_output = model.generate(**model_inputs, max_new_tokens=20, logits_processor=[logits_http_mask], do_sample=False, output_logits=True, return_dict_in_generate=True)
decoded_output = tokenizer.batch_decode(model_output.sequences, skip_special_tokens=True)[0]
print(f"--------\n{decoded_output}\n--------")

logits_http_mask.reset()
show_logits_with_color_scale(model_output, logits_http_mask)

--------
One of the most common web addresses is: http://www.google.com
--------


index,0,1,2,3,4,5,6,7,8
token,,http,://,www,.,google,.,com,<|im_end|>
'<|im_end|>',3.64,1.86,-2.00,-7.35,-2.30,-7.81,-2.93,-9.08,6.47
'.',-1.96,-3.29,2.55,-0.58,16.97,-11.73,11.51,-9.38,5.21
':',-2.43,-8.68,12.68,-5.34,-3.27,-10.44,0.23,-8.77,0.10
' ',15.35,1.52,1.08,-0.35,-1.33,-1.75,-1.85,-2.33,2.97
'es',-9.38,-9.11,-9.60,4.96,2.55,5.39,-8.35,11.78,-9.03
'com',-1.24,-3.78,-7.46,7.04,8.37,8.10,0.89,18.71,-5.12
'de',-7.68,-10.51,-7.47,5.12,4.14,6.33,-6.07,12.22,-9.86
'://',-4.75,-1.67,17.51,-9.42,-6.05,-13.55,-5.48,-12.67,-3.66
'co',-7.36,-8.68,-6.99,5.73,3.45,7.56,-4.56,14.83,-10.72


Notice the difference when we include the http: in the prompt. The model can no longer choose the token `://` and this affects the rest of the output, the model putting spaces throughout the address

In [ ]:
model_inputs = tokenizer(["One of the most common web addresses is: http:"], return_tensors="pt").to(device)
model_output = model.generate(**model_inputs, max_new_tokens=20, do_sample=False, output_logits=True, return_dict_in_generate=True)
decoded_output = tokenizer.batch_decode(model_output.sequences, skip_special_tokens=True)[0]
print(f"--------\n{decoded_output}\n--------")


--------
One of the most common web addresses is: http: // www. google. com . This is the address of the Google search engine.

Another
--------


If we enforce constrained generation, the model completes the web address without spaces even though it needs to split the tokens `:` and `//` which it likely hasn't seen before in training.

Note here we need to backtrack the `first_in_sequence` to start with the `http` token in the input. This is dependent on the tokenizer, and again this code is just for exploration and learning, don't do this in real code!

In [ ]:
logits_http_mask = RegexMask(regex_http, tokenizer, subtract_on_mask=10)
model_inputs = tokenizer(["One of the most common web addresses is: http:"], return_tensors="pt").to(device)
logits_http_mask.first_in_sequence = model_inputs.input_ids.shape[1]-2
model_output = model.generate(**model_inputs, max_new_tokens=20, logits_processor=[logits_http_mask], do_sample=False, output_logits=True, return_dict_in_generate=True)
decoded_output = tokenizer.batch_decode(model_output.sequences, skip_special_tokens=True)[0]
print(f"--------\n{decoded_output}\n--------")

logits_http_mask.first_in_sequence = model_inputs.input_ids.shape[1]-2
show_logits_with_color_scale(model_output, logits_http_mask)

--------
One of the most common web addresses is: http://www.google.com/

--------


index,0,1,2,3,4,5,6,7,8
token,//,www,.,google,.,com,/,,<|im_end|>
'<|im_end|>',-1.63,-2.78,-4.37,-8.76,-3.09,-8.32,5.14,-1.07,8.03
'.',-3.56,0.81,16.26,-11.55,19.62,-9.27,5.27,2.69,-0.50
'/',7.15,-2.25,-3.11,-8.91,10.64,-6.79,15.97,2.59,-1.98
'\n',3.11,2.90,-1.35,-0.81,12.07,0.30,15.21,11.93,5.90
'es',-2.52,5.12,3.41,5.72,2.16,10.90,2.02,5.22,-8.68
'com',-2.44,8.66,7.49,7.63,10.63,17.21,5.24,7.09,-6.38
'de',-6.49,4.26,4.39,6.33,4.09,11.15,1.57,3.69,-8.71
'//',13.28,-4.55,-4.76,-6.57,9.53,-7.53,10.08,4.42,-1.01
'co',-5.05,5.04,3.63,7.22,5.41,13.03,-0.55,3.76,-7.96


With constrained generation we are not restricted to greedy sampling, we can use multi-nomial sampling of the masked logits to generate different outputs:

In [ ]:
logits_http_mask = RegexMask(regex_http, tokenizer, subtract_on_mask=torch.inf)
model_inputs = tokenizer(["One of the most common web addresses is: "], return_tensors="pt").to(device)

for ii in range(10):
    logits_http_mask.reset()
    model_output = model.generate(**model_inputs, max_new_tokens=20, logits_processor=[logits_http_mask], do_sample=True, output_logits=True, return_dict_in_generate=True)
    print(tokenizer.batch_decode(model_output.sequences, skip_special_tokens=True)[0])

One of the most common web addresses is:  http://www.google.com/

One of the most common web addresses is:  http://example.com/.

One of the most common web addresses is:  http://www.example.com

One of the most common web addresses is:  http://www.google.com/.

One of the most common web addresses is:  http://www.example.com

One of the most common web addresses is:  http://www.google.com

One of the most common web addresses is:  https://www.google.com/

One of the most common web addresses is:  https://www.google.com/about

One of the most common web addresses is:  http://username.net/your/document

One of the most common web addresses is:  http://www.example.com

